In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/FC26_20250921.csv")
df.drop(columns=["fifa_version", "fifa_update", "fifa_update_date", "player_face_url", "work_rate", "player_url"], inplace=True)

df_draft = df[df['overall'] >= 75].copy()

faixas = [
        (df_draft['overall'] >= 87),
        (df_draft['overall'] >= 84) & (df_draft['overall'] < 87),
        (df_draft['overall'] >= 80) & (df_draft['overall'] < 84),
        (df_draft['overall'] < 80)
]
    
pesos = [5, 15, 40, 80]
df_draft['weight'] = np.select(faixas, pesos, default=0)

/tmp/ipykernel_37441/387547489.py:4: DtypeWarning: Columns (0: player_tags) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/FC26_20250921.csv")


In [3]:
from draft import generate_draft_pack
from bot import Bot
from eafc_utils import f

print("===========================================\n INICIANDO SIMULADOR DE DRAFT EA FC \n===========================================")

# 4-3-3
formacao_433 = ['ST', 'LW', 'RW', 'CM', 'CM', 'CM', 'LB', 'CB', 'CB', 'RB', 'GK']
remaining_positions = list(formacao_433) # Deep copy 
meu_time = []
ids_escolhidos = set()

NUMERO_DE_ROLLOUTS = 2000

meu_bot = Bot(mode="expectimax", num_rollouts=NUMERO_DE_ROLLOUTS)

for rodada in range(1, 12):
    print(f"\n--- RODADA {rodada}/11 ---")
    pacote = generate_draft_pack(df_draft, rodada, remaining_positions, ids_escolhidos)
    opcoes = pacote.reset_index(drop=True)
    print(opcoes[['short_name', 'overall', 'club_name', 'league_name', 'nationality_name']])

    df_meu_time = pd.DataFrame(meu_time) if meu_time else pd.DataFrame(columns=df_draft.columns)

    indice_escolhido, posicao_escolhida = meu_bot.choose(
        options=opcoes,
        current_squad=df_meu_time,
        db=df_draft,
        current_round=rodada,
        remaining_positions=remaining_positions
    )

    carta_escolhida = opcoes.iloc[indice_escolhido].copy()
    carta_escolhida['choosen_position'] = posicao_escolhida
    remaining_positions.remove(posicao_escolhida)
    meu_time.append(carta_escolhida)
    ids_escolhidos.add(carta_escolhida['player_id'])

    print(f">>> ESCOLHA DO BOT: {carta_escolhida['short_name']} (OVR {carta_escolhida['overall']})")

print("\n===========================================")
print(" DRAFT CONCLUÍDO! SEU ELENCO FINAL: ")
print("===========================================")
df_final = pd.DataFrame(meu_time)
print(df_final[['short_name', 'overall', 'league_name', 'nationality_name', 'club_name']])
nota_final = f(df_final)
print(f"\nNOTA FINAL DO TIME:", int(nota_final))

 INICIANDO SIMULADOR DE DRAFT EA FC 

--- RODADA 1/11 ---
     short_name  overall            club_name league_name nationality_name
0       H. Kane       89    FC Bayern München  Bundesliga          England
1    O. Dembélé       90  Paris Saint-Germain     Ligue 1           France
2    J. Kimmich       89    FC Bayern München  Bundesliga          Germany
3      Vini Jr.       89          Real Madrid     La Liga           Brazil
4  Lamine Yamal       89         FC Barcelona     La Liga            Spain
>>> ESCOLHA DO BOT: Vini Jr. (OVR 89)

--- RODADA 2/11 ---
       short_name  overall          club_name     league_name nationality_name
0   B. Chardonnet       77  Stade Brestois 29         Ligue 1           France
1      S. Mijnans       75         AZ Alkmaar      Eredivisie      Netherlands
2       T. Almada       79    Atlético Madrid         La Liga        Argentina
3  L. Klostermann       78         RB Leipzig      Bundesliga          Germany
4       B. Davies       75  Tottenham 

In [7]:
from bot import SimulatedAnnealingBot
from eafc_utils import f

formacao_433 = ['ST', 'LW', 'RW', 'CM', 'CM', 'CM', 'LB', 'CB', 'CB', 'RB', 'GK']

sa_bot = SimulatedAnnealingBot(temp_inicial=100, temp_final=0.1, alpha=0.995, iteracoes=5000)
df_sa = sa_bot.montar_time(df_draft, formacao_433)

print("===========================================")
print(" SIMULATED ANNEALING - ELENCO FINAL ")
print("===========================================")
print(df_sa[['short_name', 'overall', 'choosen_position', 'league_name', 'nationality_name', 'club_name']])
print(f"\nNOTA FINAL DO TIME: {int(f(df_sa))}")

 SIMULATED ANNEALING - ELENCO FINAL 
        short_name  overall choosen_position     league_name nationality_name  \
0       Iago Aspas       83               ST         La Liga            Spain   
1         P. Foden       85               LW  Premier League          England   
2         A. Güler       81               RW         La Liga          Türkiye   
3     E. Camavinga       83               CM         La Liga           France   
4    Pablo Barrios       82               CM         La Liga            Spain   
5       João Gomes       78               CM  Premier League           Brazil   
6   Marc Cucurella       84               LB  Premier League            Spain   
7       B. Diakité       79               CB  Premier League           France   
8       L. Cabrera       75               CB         La Liga          Uruguay   
9      T. Chalobah       79               RB  Premier League          England   
10    P. Gazzaniga       79               GK         La Liga        Arge